In [1]:
# ============================================================
# Day 4 - Step 1: Load Dataset
# ============================================================

import os
import pandas as pd
import numpy as np

DATA_DIR = "../data/raw"
BASE_FILE = "Base.csv"
TARGET = "fraud_bool"

base_df = pd.read_csv(
    os.path.join(DATA_DIR, BASE_FILE)
)

print("Dataset loaded successfully.")
print("Dataset:", BASE_FILE)
print("Shape:", base_df.shape)
print("Target:", TARGET)

Dataset loaded successfully.
Dataset: Base.csv
Shape: (1000000, 32)
Target: fraud_bool


In [2]:
# ============================================================
# Day 4 - Step 2: Separate Features and Target
# ============================================================

X = base_df.drop(columns=[TARGET])
y = base_df[TARGET]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nTarget percentage:")
print((y.value_counts(normalize=True) * 100).round(2))

Features shape: (1000000, 31)
Target shape: (1000000,)

Target distribution:
fraud_bool
0    988971
1     11029
Name: count, dtype: int64

Target percentage:
fraud_bool
0    98.9
1     1.1
Name: proportion, dtype: float64


In [3]:
# ============================================================
# Day 4 - Step 3: Remove Constant Feature
# ============================================================

constant_features = [
    col for col in X.columns
    if X[col].nunique() <= 1
]

print("Constant features found:")
print(constant_features)

X = X.drop(columns=constant_features)

print("\nFeatures after removing constant features:", X.shape[1])
print("X shape:", X.shape)

Constant features found:
['device_fraud_count']

Features after removing constant features: 30
X shape: (1000000, 30)


In [4]:
# ============================================================
# Day 4 - Step 4: Stratified Train/Test Split
# ============================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train/Test split completed.")

print("\nTraining data:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nTesting data:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTesting target distribution:")
print(y_test.value_counts())
print((y_test.value_counts(normalize=True) * 100).round(2))

Train/Test split completed.

Training data:
X_train: (800000, 30)
y_train: (800000,)

Testing data:
X_test: (200000, 30)
y_test: (200000,)

Training target distribution:
fraud_bool
0    791177
1      8823
Name: count, dtype: int64
fraud_bool
0    98.9
1     1.1
Name: proportion, dtype: float64

Testing target distribution:
fraud_bool
0    197794
1      2206
Name: count, dtype: int64
fraud_bool
0    98.9
1     1.1
Name: proportion, dtype: float64


In [5]:
# ============================================================
# Day 4 - Step 5: Identify Categorical Features
# ============================================================

categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_features = X_train.select_dtypes(
    exclude=["object", "category"]
).columns.tolist()

print("Categorical features:")
print(categorical_features)

print("\nNumber of categorical features:", len(categorical_features))

print("\nNumerical features:")
print(numerical_features)

print("\nNumber of numerical features:", len(numerical_features))

Categorical features:
['payment_type', 'employment_status', 'housing_status', 'source', 'device_os']

Number of categorical features: 5

Numerical features:
['income', 'name_email_similarity', 'prev_address_months_count', 'current_address_months_count', 'customer_age', 'days_since_request', 'intended_balcon_amount', 'zip_count_4w', 'velocity_6h', 'velocity_24h', 'velocity_4w', 'bank_branch_count_8w', 'date_of_birth_distinct_emails_4w', 'credit_risk_score', 'email_is_free', 'phone_home_valid', 'phone_mobile_valid', 'bank_months_count', 'has_other_cards', 'proposed_credit_limit', 'foreign_request', 'session_length_in_minutes', 'keep_alive_session', 'device_distinct_emails_8w', 'month']

Number of numerical features: 25


C:\Users\prala\AppData\Local\Temp\ipykernel_8756\14035773.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


In [6]:
# ============================================================
# Day 4 - Step 6: Inspect Categorical Categories
# ============================================================

for col in categorical_features:
    print(f"\n{col}")
    print("Number of categories:", X_train[col].nunique())
    print("Categories:", sorted(X_train[col].dropna().unique()))


payment_type
Number of categories: 5
Categories: ['AA', 'AB', 'AC', 'AD', 'AE']

employment_status
Number of categories: 7
Categories: ['CA', 'CB', 'CC', 'CD', 'CE', 'CF', 'CG']

housing_status
Number of categories: 7
Categories: ['BA', 'BB', 'BC', 'BD', 'BE', 'BF', 'BG']

source
Number of categories: 2
Categories: ['INTERNET', 'TELEAPP']

device_os
Number of categories: 5
Categories: ['linux', 'macintosh', 'other', 'windows', 'x11']


In [7]:
# ============================================================
# Day 4 - Step 7: One-Hot Encode Categorical Features
# ============================================================

from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# Fit only on training data
encoder.fit(X_train[categorical_features])

# Transform categorical features
X_train_cat = encoder.transform(X_train[categorical_features])
X_test_cat = encoder.transform(X_test[categorical_features])

print("One-hot encoding completed.")

print("\nEncoded training shape:", X_train_cat.shape)
print("Encoded testing shape:", X_test_cat.shape)

print("\nNumber of encoded features:", X_train_cat.shape[1])

One-hot encoding completed.

Encoded training shape: (800000, 26)
Encoded testing shape: (200000, 26)

Number of encoded features: 26


In [8]:
# ============================================================
# Day 4 - Step 8: Combine Numerical and Categorical Features
# ============================================================

# Convert numerical features to NumPy arrays
X_train_num = X_train[numerical_features].to_numpy()
X_test_num = X_test[numerical_features].to_numpy()

# Combine numerical + encoded categorical features
X_train_processed = np.hstack([
    X_train_num,
    X_train_cat
])

X_test_processed = np.hstack([
    X_test_num,
    X_test_cat
])

print("Feature combination completed.")

print("\nProcessed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

print("\nTotal model features:", X_train_processed.shape[1])

Feature combination completed.

Processed training shape: (800000, 51)
Processed testing shape: (200000, 51)

Total model features: 51


In [9]:
# ============================================================
# Day 4 - Step 9: Train Baseline Logistic Regression
# ============================================================

from sklearn.linear_model import LogisticRegression

baseline_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

print("Training baseline Logistic Regression...")

baseline_model.fit(
    X_train_processed,
    y_train
)

print("Baseline model training completed.")

Training baseline Logistic Regression...
Baseline model training completed.


C:\Users\prala\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [10]:
# ============================================================
# Day 4 - Step 10: Scale Numerical Features
# ============================================================

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit only on training numerical features
X_train_num_scaled = scaler.fit_transform(
    X_train[numerical_features]
)

# Transform test numerical features
X_test_num_scaled = scaler.transform(
    X_test[numerical_features]
)

print("Numerical feature scaling completed.")

print("\nScaled training shape:", X_train_num_scaled.shape)
print("Scaled testing shape:", X_test_num_scaled.shape)

Numerical feature scaling completed.

Scaled training shape: (800000, 25)
Scaled testing shape: (200000, 25)


In [11]:
# ============================================================
# Day 4 - Step 11: Combine Scaled + Encoded Features
# ============================================================

X_train_final = np.hstack([
    X_train_num_scaled,
    X_train_cat
])

X_test_final = np.hstack([
    X_test_num_scaled,
    X_test_cat
])

print("Final feature matrices created.")

print("\nTraining shape:", X_train_final.shape)
print("Testing shape:", X_test_final.shape)

print("\nTotal features:", X_train_final.shape[1])

Final feature matrices created.

Training shape: (800000, 51)
Testing shape: (200000, 51)

Total features: 51


In [12]:
# ============================================================
# Day 4 - Step 12: Train Scaled Logistic Regression Baseline
# ============================================================

baseline_model = LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    random_state=42
)

print("Training scaled Logistic Regression...")

baseline_model.fit(
    X_train_final,
    y_train
)

print("Scaled Logistic Regression training completed.")
print("Iterations used:", baseline_model.n_iter_[0])

Training scaled Logistic Regression...
Scaled Logistic Regression training completed.
Iterations used: 42


In [13]:
# ============================================================
# Day 4 - Step 13: Generate Baseline Predictions
# ============================================================

# Predict class labels
y_pred = baseline_model.predict(X_test_final)

# Predict fraud probabilities
y_pred_proba = baseline_model.predict_proba(X_test_final)[:, 1]

print("Predictions generated successfully.")

print("\nPredicted classes:")
print(pd.Series(y_pred).value_counts())

print("\nFirst 10 fraud probabilities:")
print(y_pred_proba[:10])

Predictions generated successfully.

Predicted classes:
0    159640
1     40360
Name: count, dtype: int64

First 10 fraud probabilities:
[0.19095071 0.43390372 0.35260807 0.1958591  0.04995757 0.72375874
 0.70855082 0.1925555  0.72174493 0.04422311]


In [14]:
# ============================================================
# Day 4 - Step 14: Baseline Model Evaluation
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)
pr_auc = average_precision_score(y_test, y_pred_proba)

print("Baseline Logistic Regression Results")
print("=" * 45)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")
print(f"PR-AUC   : {pr_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Baseline Logistic Regression Results
Accuracy : 0.8047
Precision: 0.0434
Recall   : 0.7937
F1 Score : 0.0823
ROC-AUC  : 0.8759
PR-AUC   : 0.1378

Confusion Matrix:
[[159185  38609]
 [   455   1751]]


In [15]:
# ============================================================
# Day 4 - Step 15: Store Baseline Results
# ============================================================

baseline_results = {
    "Model": "Logistic Regression",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1 Score": f1,
    "ROC-AUC": roc_auc,
    "PR-AUC": pr_auc
}

baseline_results_df = pd.DataFrame([baseline_results])

print("Baseline results stored successfully.")
print()
print(baseline_results_df.round(4))

Baseline results stored successfully.

                 Model  Accuracy  Precision  Recall  F1 Score  ROC-AUC  PR-AUC
0  Logistic Regression    0.8047     0.0434  0.7937    0.0823   0.8759  0.1378


In [16]:
# ============================================================
# Day 4 - Step 16: Train Random Forest Baseline
# ============================================================

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("Training Random Forest...")

rf_model.fit(
    X_train_processed,
    y_train
)

print("Random Forest training completed.")

Training Random Forest...
Random Forest training completed.


In [17]:
# ============================================================
# Day 4 - Step 17: Generate Random Forest Predictions
# ============================================================

# Predict class labels
rf_y_pred = rf_model.predict(X_test_processed)

# Predict fraud probabilities
rf_y_pred_proba = rf_model.predict_proba(X_test_processed)[:, 1]

print("Random Forest predictions generated successfully.")

print("\nPredicted classes:")
print(pd.Series(rf_y_pred).value_counts())

print("\nFirst 10 fraud probabilities:")
print(rf_y_pred_proba[:10])

Random Forest predictions generated successfully.

Predicted classes:
0    199854
1       146
Name: count, dtype: int64

First 10 fraud probabilities:
[0.01 0.   0.01 0.04 0.02 0.02 0.15 0.   0.   0.  ]


In [18]:
# ============================================================
# Day 4 - Step 18: Random Forest Evaluation
# ============================================================

rf_accuracy = accuracy_score(y_test, rf_y_pred)
rf_precision = precision_score(y_test, rf_y_pred)
rf_recall = recall_score(y_test, rf_y_pred)
rf_f1 = f1_score(y_test, rf_y_pred)
rf_roc_auc = roc_auc_score(y_test, rf_y_pred_proba)
rf_pr_auc = average_precision_score(y_test, rf_y_pred_proba)

print("Random Forest Results")
print("=" * 45)

print(f"Accuracy : {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall   : {rf_recall:.4f}")
print(f"F1 Score : {rf_f1:.4f}")
print(f"ROC-AUC  : {rf_roc_auc:.4f}")
print(f"PR-AUC   : {rf_pr_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_y_pred))

Random Forest Results
Accuracy : 0.9889
Precision: 0.4247
Recall   : 0.0281
F1 Score : 0.0527
ROC-AUC  : 0.8577
PR-AUC   : 0.1207

Confusion Matrix:
[[197710     84]
 [  2144     62]]


In [19]:
# ============================================================
# Day 4 - Step 19: Store Random Forest Results
# ============================================================

rf_results = {
    "Model": "Random Forest",
    "Accuracy": rf_accuracy,
    "Precision": rf_precision,
    "Recall": rf_recall,
    "F1 Score": rf_f1,
    "ROC-AUC": rf_roc_auc,
    "PR-AUC": rf_pr_auc
}

# Add Random Forest to the existing results
baseline_results_df = pd.concat(
    [
        baseline_results_df,
        pd.DataFrame([rf_results])
    ],
    ignore_index=True
)

print("Random Forest results stored successfully.")
print()
print(baseline_results_df.round(4))

Random Forest results stored successfully.

                 Model  Accuracy  Precision  Recall  F1 Score  ROC-AUC  PR-AUC
0  Logistic Regression    0.8047     0.0434  0.7937    0.0823   0.8759  0.1378
1        Random Forest    0.9889     0.4247  0.0281    0.0527   0.8577  0.1207


In [20]:
# ============================================================
# Day 4 - Step 20: Compare Baseline Models
# ============================================================

model_comparison = baseline_results_df.sort_values(
    by="PR-AUC",
    ascending=False
).reset_index(drop=True)

print("Baseline Model Comparison")
print("=" * 60)
print(model_comparison.round(4))

Baseline Model Comparison
                 Model  Accuracy  Precision  Recall  F1 Score  ROC-AUC  PR-AUC
0  Logistic Regression    0.8047     0.0434  0.7937    0.0823   0.8759  0.1378
1        Random Forest    0.9889     0.4247  0.0281    0.0527   0.8577  0.1207


In [21]:
# ============================================================
# Day 5 - Step 1: Train Unweighted Logistic Regression
# ============================================================

unweighted_lr = LogisticRegression(
    class_weight=None,
    max_iter=2000,
    random_state=42
)

print("Training unweighted Logistic Regression...")

unweighted_lr.fit(
    X_train_final,
    y_train
)

print("Unweighted Logistic Regression training completed.")
print("Iterations used:", unweighted_lr.n_iter_[0])

Training unweighted Logistic Regression...
Unweighted Logistic Regression training completed.
Iterations used: 23


In [22]:
# ============================================================
# Day 5 - Step 2: Evaluate Unweighted Logistic Regression
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

unweighted_y_pred = unweighted_lr.predict(X_test_final)
unweighted_y_pred_proba = unweighted_lr.predict_proba(X_test_final)[:, 1]

unweighted_accuracy = accuracy_score(y_test, unweighted_y_pred)
unweighted_precision = precision_score(y_test, unweighted_y_pred)
unweighted_recall = recall_score(y_test, unweighted_y_pred)
unweighted_f1 = f1_score(y_test, unweighted_y_pred)
unweighted_roc_auc = roc_auc_score(y_test, unweighted_y_pred_proba)
unweighted_pr_auc = average_precision_score(
    y_test,
    unweighted_y_pred_proba
)

print("Unweighted Logistic Regression")
print("=" * 50)
print(f"Accuracy : {unweighted_accuracy:.4f}")
print(f"Precision: {unweighted_precision:.4f}")
print(f"Recall   : {unweighted_recall:.4f}")
print(f"F1 Score : {unweighted_f1:.4f}")
print(f"ROC-AUC  : {unweighted_roc_auc:.4f}")
print(f"PR-AUC   : {unweighted_pr_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, unweighted_y_pred))

Unweighted Logistic Regression
Accuracy : 0.9890
Precision: 0.6383
Recall   : 0.0136
F1 Score : 0.0266
ROC-AUC  : 0.8746
PR-AUC   : 0.1433

Confusion Matrix:
[[197777     17]
 [  2176     30]]


In [23]:
# ============================================================
# Day 5 - Step 3: Store Unweighted Logistic Regression Results
# ============================================================

unweighted_results = {
    "Model": "Logistic Regression (Unweighted)",
    "Accuracy": unweighted_accuracy,
    "Precision": unweighted_precision,
    "Recall": unweighted_recall,
    "F1 Score": unweighted_f1,
    "ROC-AUC": unweighted_roc_auc,
    "PR-AUC": unweighted_pr_auc
}

baseline_results_df = pd.concat(
    [
        baseline_results_df,
        pd.DataFrame([unweighted_results])
    ],
    ignore_index=True
)

print("Updated Model Comparison")
print("=" * 70)
print(
    baseline_results_df
    .sort_values(by="PR-AUC", ascending=False)
    .reset_index(drop=True)
    .round(4)
)

Updated Model Comparison
                              Model  Accuracy  Precision  Recall  F1 Score  \
0  Logistic Regression (Unweighted)    0.9890     0.6383  0.0136    0.0266   
1               Logistic Regression    0.8047     0.0434  0.7937    0.0823   
2                     Random Forest    0.9889     0.4247  0.0281    0.0527   

   ROC-AUC  PR-AUC  
0   0.8746  0.1433  
1   0.8759  0.1378  
2   0.8577  0.1207  


In [24]:
# ============================================================
# Day 5 - Step 4: Classification Threshold Analysis
# ============================================================

thresholds = [0.50, 0.30, 0.20, 0.10, 0.05, 0.02, 0.01]

threshold_results = []

for threshold in thresholds:
    threshold_pred = (
        unweighted_y_pred_proba >= threshold
    ).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_test, threshold_pred),
        "Recall": recall_score(y_test, threshold_pred),
        "F1 Score": f1_score(y_test, threshold_pred)
    })

threshold_results_df = pd.DataFrame(threshold_results)

print("Threshold Analysis")
print("=" * 60)
print(threshold_results_df.round(4))

Threshold Analysis
   Threshold  Precision  Recall  F1 Score
0       0.50     0.6383  0.0136    0.0266
1       0.30     0.3967  0.0549    0.0964
2       0.20     0.2830  0.1197    0.1682
3       0.10     0.1769  0.2774    0.2161
4       0.05     0.1142  0.4637    0.1833
5       0.02     0.0625  0.6836    0.1146
6       0.01     0.0405  0.8182    0.0772


In [25]:
# ============================================================
# Day 5 - Step 5: HistGradientBoosting Classifier
# ============================================================

from sklearn.ensemble import HistGradientBoostingClassifier

hgb_model = HistGradientBoostingClassifier(
    max_iter=200,
    learning_rate=0.1,
    max_leaf_nodes=31,
    random_state=42
)

print("Training HistGradientBoostingClassifier...")

hgb_model.fit(
    X_train_processed,
    y_train
)

print("HistGradientBoostingClassifier training completed.")

Training HistGradientBoostingClassifier...
HistGradientBoostingClassifier training completed.


In [26]:
# ============================================================
# Day 5 - Step 6: Evaluate HistGradientBoosting
# ============================================================

hgb_y_pred = hgb_model.predict(X_test_processed)
hgb_y_pred_proba = hgb_model.predict_proba(X_test_processed)[:, 1]

hgb_accuracy = accuracy_score(y_test, hgb_y_pred)
hgb_precision = precision_score(y_test, hgb_y_pred)
hgb_recall = recall_score(y_test, hgb_y_pred)
hgb_f1 = f1_score(y_test, hgb_y_pred)
hgb_roc_auc = roc_auc_score(y_test, hgb_y_pred_proba)
hgb_pr_auc = average_precision_score(
    y_test,
    hgb_y_pred_proba
)

print("HistGradientBoostingClassifier")
print("=" * 50)
print(f"Accuracy : {hgb_accuracy:.4f}")
print(f"Precision: {hgb_precision:.4f}")
print(f"Recall   : {hgb_recall:.4f}")
print(f"F1 Score : {hgb_f1:.4f}")
print(f"ROC-AUC  : {hgb_roc_auc:.4f}")
print(f"PR-AUC   : {hgb_pr_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, hgb_y_pred))

HistGradientBoostingClassifier
Accuracy : 0.9888
Precision: 0.4188
Recall   : 0.0444
F1 Score : 0.0803
ROC-AUC  : 0.8931
PR-AUC   : 0.1542

Confusion Matrix:
[[197658    136]
 [  2108     98]]


In [27]:
# ============================================================
# Day 5 - Step 7: HGB Classification Threshold Analysis
# ============================================================

hgb_thresholds = [0.50, 0.30, 0.20, 0.10, 0.05, 0.02, 0.01]

hgb_threshold_results = []

for threshold in hgb_thresholds:
    threshold_pred = (
        hgb_y_pred_proba >= threshold
    ).astype(int)

    hgb_threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_test, threshold_pred),
        "Recall": recall_score(y_test, threshold_pred),
        "F1 Score": f1_score(y_test, threshold_pred)
    })

hgb_threshold_results_df = pd.DataFrame(
    hgb_threshold_results
)

print("HistGradientBoosting Threshold Analysis")
print("=" * 60)
print(hgb_threshold_results_df.round(4))

HistGradientBoosting Threshold Analysis
   Threshold  Precision  Recall  F1 Score
0       0.50     0.4188  0.0444    0.0803
1       0.30     0.3308  0.0970    0.1500
2       0.20     0.2733  0.1623    0.2036
3       0.10     0.1854  0.2969    0.2283
4       0.05     0.1282  0.4674    0.2012
5       0.02     0.0732  0.6768    0.1321
6       0.01     0.0472  0.8105    0.0891


In [28]:
# ============================================================
# Day 5 - Step 8: Store HistGradientBoosting Results
# ============================================================

hgb_results = {
    "Model": "HistGradientBoosting",
    "Accuracy": hgb_accuracy,
    "Precision": hgb_precision,
    "Recall": hgb_recall,
    "F1 Score": hgb_f1,
    "ROC-AUC": hgb_roc_auc,
    "PR-AUC": hgb_pr_auc
}

baseline_results_df = pd.concat(
    [
        baseline_results_df,
        pd.DataFrame([hgb_results])
    ],
    ignore_index=True
)

model_comparison = (
    baseline_results_df
    .sort_values(by="PR-AUC", ascending=False)
    .reset_index(drop=True)
)

print("Updated Model Comparison")
print("=" * 75)
print(model_comparison.round(4))

Updated Model Comparison
                              Model  Accuracy  Precision  Recall  F1 Score  \
0              HistGradientBoosting    0.9888     0.4188  0.0444    0.0803   
1  Logistic Regression (Unweighted)    0.9890     0.6383  0.0136    0.0266   
2               Logistic Regression    0.8047     0.0434  0.7937    0.0823   
3                     Random Forest    0.9889     0.4247  0.0281    0.0527   

   ROC-AUC  PR-AUC  
0   0.8931  0.1542  
1   0.8746  0.1433  
2   0.8759  0.1378  
3   0.8577  0.1207  


In [29]:
# ============================================================
# Day 5 - Step 9: Compare Best Threshold Results
# ============================================================

comparison_threshold = 0.10

lr_threshold_pred = (
    unweighted_y_pred_proba >= comparison_threshold
).astype(int)

hgb_threshold_pred = (
    hgb_y_pred_proba >= comparison_threshold
).astype(int)

threshold_model_comparison = pd.DataFrame([
    {
        "Model": "Logistic Regression (Unweighted)",
        "Threshold": comparison_threshold,
        "Precision": precision_score(y_test, lr_threshold_pred),
        "Recall": recall_score(y_test, lr_threshold_pred),
        "F1 Score": f1_score(y_test, lr_threshold_pred)
    },
    {
        "Model": "HistGradientBoosting",
        "Threshold": comparison_threshold,
        "Precision": precision_score(y_test, hgb_threshold_pred),
        "Recall": recall_score(y_test, hgb_threshold_pred),
        "F1 Score": f1_score(y_test, hgb_threshold_pred)
    }
])

print("Model Comparison at Threshold 0.10")
print("=" * 65)
print(threshold_model_comparison.round(4))

Model Comparison at Threshold 0.10
                              Model  Threshold  Precision  Recall  F1 Score
0  Logistic Regression (Unweighted)        0.1     0.1769  0.2774    0.2161
1              HistGradientBoosting        0.1     0.1854  0.2969    0.2283


In [31]:
# ============================================================
# Day 5 - Step 10: Tuned HistGradientBoosting
# ============================================================

tuned_hgb_model = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    random_state=42
)

print("Training Tuned HistGradientBoostingClassifier...")

tuned_hgb_model.fit(
    X_train_processed,
    y_train
)

print("Tuned HistGradientBoostingClassifier training completed.")

Training Tuned HistGradientBoostingClassifier...
Tuned HistGradientBoostingClassifier training completed.


In [32]:
# ============================================================
# Day 5 - Step 11: Evaluate Tuned HistGradientBoosting
# ============================================================

tuned_hgb_y_pred = tuned_hgb_model.predict(X_test_processed)
tuned_hgb_y_pred_proba = tuned_hgb_model.predict_proba(
    X_test_processed
)[:, 1]

tuned_hgb_accuracy = accuracy_score(
    y_test,
    tuned_hgb_y_pred
)

tuned_hgb_precision = precision_score(
    y_test,
    tuned_hgb_y_pred
)

tuned_hgb_recall = recall_score(
    y_test,
    tuned_hgb_y_pred
)

tuned_hgb_f1 = f1_score(
    y_test,
    tuned_hgb_y_pred
)

tuned_hgb_roc_auc = roc_auc_score(
    y_test,
    tuned_hgb_y_pred_proba
)

tuned_hgb_pr_auc = average_precision_score(
    y_test,
    tuned_hgb_y_pred_proba
)

tuned_hgb_cm = confusion_matrix(
    y_test,
    tuned_hgb_y_pred
)

print("Tuned HistGradientBoosting Results")
print("=" * 65)
print(f"Accuracy : {tuned_hgb_accuracy:.4f}")
print(f"Precision: {tuned_hgb_precision:.4f}")
print(f"Recall   : {tuned_hgb_recall:.4f}")
print(f"F1 Score : {tuned_hgb_f1:.4f}")
print(f"ROC-AUC  : {tuned_hgb_roc_auc:.4f}")
print(f"PR-AUC   : {tuned_hgb_pr_auc:.4f}")

print("\nConfusion Matrix:")
print(tuned_hgb_cm)

Tuned HistGradientBoosting Results
Accuracy : 0.9890
Precision: 0.5395
Recall   : 0.0372
F1 Score : 0.0696
ROC-AUC  : 0.8961
PR-AUC   : 0.1698

Confusion Matrix:
[[197724     70]
 [  2124     82]]


In [33]:
# ============================================================
# Day 5 - Step 12: Tuned HGB Threshold Analysis
# ============================================================

thresholds = [0.50, 0.30, 0.20, 0.10, 0.05, 0.02, 0.01]

tuned_hgb_threshold_results = []

for threshold in thresholds:

    threshold_pred = (
        tuned_hgb_y_pred_proba >= threshold
    ).astype(int)

    tuned_hgb_threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_test,
            threshold_pred
        ),
        "Recall": recall_score(
            y_test,
            threshold_pred
        ),
        "F1 Score": f1_score(
            y_test,
            threshold_pred
        )
    })

tuned_hgb_threshold_df = pd.DataFrame(
    tuned_hgb_threshold_results
)

print("Tuned HGB Threshold Analysis")
print("=" * 60)
print(tuned_hgb_threshold_df.round(4))

Tuned HGB Threshold Analysis
   Threshold  Precision  Recall  F1 Score
0       0.50     0.5395  0.0372    0.0696
1       0.30     0.3880  0.1052    0.1655
2       0.20     0.2908  0.1727    0.2167
3       0.10     0.1904  0.3128    0.2367
4       0.05     0.1288  0.4859    0.2037
5       0.02     0.0732  0.6908    0.1324
6       0.01     0.0474  0.8191    0.0895


In [34]:
# ============================================================
# Day 5 - Step 13: Original vs Tuned HGB Comparison
# ============================================================

thresholds = [0.50, 0.30, 0.20, 0.10, 0.05, 0.02, 0.01]

hgb_comparison_results = []

for threshold in thresholds:

    original_pred = (
        hgb_y_pred_proba >= threshold
    ).astype(int)

    tuned_pred = (
        tuned_hgb_y_pred_proba >= threshold
    ).astype(int)

    hgb_comparison_results.append({
        "Threshold": threshold,

        "Original HGB Precision": precision_score(
            y_test, original_pred
        ),
        "Original HGB Recall": recall_score(
            y_test, original_pred
        ),
        "Original HGB F1": f1_score(
            y_test, original_pred
        ),

        "Tuned HGB Precision": precision_score(
            y_test, tuned_pred
        ),
        "Tuned HGB Recall": recall_score(
            y_test, tuned_pred
        ),
        "Tuned HGB F1": f1_score(
            y_test, tuned_pred
        )
    })

hgb_comparison_df = pd.DataFrame(
    hgb_comparison_results
)

print("Original vs Tuned HGB")
print("=" * 100)
print(hgb_comparison_df.round(4))

Original vs Tuned HGB
   Threshold  Original HGB Precision  Original HGB Recall  Original HGB F1  \
0       0.50                  0.4188               0.0444           0.0803   
1       0.30                  0.3308               0.0970           0.1500   
2       0.20                  0.2733               0.1623           0.2036   
3       0.10                  0.1854               0.2969           0.2283   
4       0.05                  0.1282               0.4674           0.2012   
5       0.02                  0.0732               0.6768           0.1321   
6       0.01                  0.0472               0.8105           0.0891   

   Tuned HGB Precision  Tuned HGB Recall  Tuned HGB F1  
0               0.5395            0.0372        0.0696  
1               0.3880            0.1052        0.1655  
2               0.2908            0.1727        0.2167  
3               0.1904            0.3128        0.2367  
4               0.1288            0.4859        0.2037  
5          

In [35]:
# ============================================================
# Day 5 - Step 14: Store Tuned HGB Results
# ============================================================

tuned_hgb_results = {
    "Model": "Tuned HistGradientBoosting",
    "Accuracy": tuned_hgb_accuracy,
    "Precision": tuned_hgb_precision,
    "Recall": tuned_hgb_recall,
    "F1 Score": tuned_hgb_f1,
    "ROC-AUC": tuned_hgb_roc_auc,
    "PR-AUC": tuned_hgb_pr_auc
}

baseline_results_df = pd.concat(
    [
        baseline_results_df,
        pd.DataFrame([tuned_hgb_results])
    ],
    ignore_index=True
)

final_model_comparison = (
    baseline_results_df
    .sort_values(by="PR-AUC", ascending=False)
    .reset_index(drop=True)
)

print("Final Model Comparison")
print("=" * 80)
print(final_model_comparison.round(4))

Final Model Comparison
                              Model  Accuracy  Precision  Recall  F1 Score  \
0        Tuned HistGradientBoosting    0.9890     0.5395  0.0372    0.0696   
1              HistGradientBoosting    0.9888     0.4188  0.0444    0.0803   
2  Logistic Regression (Unweighted)    0.9890     0.6383  0.0136    0.0266   
3               Logistic Regression    0.8047     0.0434  0.7937    0.0823   
4                     Random Forest    0.9889     0.4247  0.0281    0.0527   

   ROC-AUC  PR-AUC  
0   0.8961  0.1698  
1   0.8931  0.1542  
2   0.8746  0.1433  
3   0.8759  0.1378  
4   0.8577  0.1207  


In [36]:
# ============================================================
# Day 5 - Step 15: Final Model Selection
# ============================================================

final_model = tuned_hgb_model
final_threshold = 0.10

final_y_pred = (
    tuned_hgb_y_pred_proba >= final_threshold
).astype(int)

final_accuracy = accuracy_score(
    y_test,
    final_y_pred
)

final_precision = precision_score(
    y_test,
    final_y_pred
)

final_recall = recall_score(
    y_test,
    final_y_pred
)

final_f1 = f1_score(
    y_test,
    final_y_pred
)

final_roc_auc = roc_auc_score(
    y_test,
    tuned_hgb_y_pred_proba
)

final_pr_auc = average_precision_score(
    y_test,
    tuned_hgb_y_pred_proba
)

final_cm = confusion_matrix(
    y_test,
    final_y_pred
)

print("FINAL MODEL")
print("=" * 60)
print("Model     : Tuned HistGradientBoosting")
print("Threshold :", final_threshold)

print("\nFinal Metrics")
print("-" * 60)
print(f"Accuracy  : {final_accuracy:.4f}")
print(f"Precision : {final_precision:.4f}")
print(f"Recall    : {final_recall:.4f}")
print(f"F1 Score  : {final_f1:.4f}")
print(f"ROC-AUC   : {final_roc_auc:.4f}")
print(f"PR-AUC    : {final_pr_auc:.4f}")

print("\nConfusion Matrix")
print("-" * 60)
print(final_cm)

FINAL MODEL
Model     : Tuned HistGradientBoosting
Threshold : 0.1

Final Metrics
------------------------------------------------------------
Accuracy  : 0.9778
Precision : 0.1904
Recall    : 0.3128
F1 Score  : 0.2367
ROC-AUC   : 0.8961
PR-AUC    : 0.1698

Confusion Matrix
------------------------------------------------------------
[[194861   2933]
 [  1516    690]]


In [37]:
# ============================================================
# Day 5 - Step 16: Final Evaluation Summary
# ============================================================

final_evaluation = pd.DataFrame([
    {
        "Model": "Tuned HistGradientBoosting",
        "Threshold": final_threshold,
        "Accuracy": final_accuracy,
        "Precision": final_precision,
        "Recall": final_recall,
        "F1 Score": final_f1,
        "ROC-AUC": final_roc_auc,
        "PR-AUC": final_pr_auc,
        "True Negatives": final_cm[0, 0],
        "False Positives": final_cm[0, 1],
        "False Negatives": final_cm[1, 0],
        "True Positives": final_cm[1, 1]
    }
])

print("Final Evaluation Summary")
print("=" * 100)
print(final_evaluation.round(4))

Final Evaluation Summary
                        Model  Threshold  Accuracy  Precision  Recall  \
0  Tuned HistGradientBoosting        0.1    0.9778     0.1904  0.3128   

   F1 Score  ROC-AUC  PR-AUC  True Negatives  False Positives  \
0    0.2367   0.8961  0.1698          194861             2933   

   False Negatives  True Positives  
0             1516             690  


In [38]:
# ============================================================
# Day 5 - Step 17: Save Final Evaluation Results
# ============================================================

OUTPUT_FILE = "../final_model_evaluation.csv"

final_evaluation.to_csv(
    OUTPUT_FILE,
    index=False
)

print(f"Final evaluation saved successfully to: {OUTPUT_FILE}")

Final evaluation saved successfully to: ../final_model_evaluation.csv


In [3]:
import os
import pandas as pd
import numpy as np

DATA_DIR = "../data/raw"
BASE_FILE = "Base.csv"
TARGET = "fraud_bool"

base_df = pd.read_csv(
    os.path.join(DATA_DIR, BASE_FILE)
)

print("Dataset loaded successfully")
print("Shape:", base_df.shape)
print("Target distribution:")
print(base_df[TARGET].value_counts())

Dataset loaded successfully
Shape: (1000000, 32)
Target distribution:
fraud_bool
0    988971
1     11029
Name: count, dtype: int64


In [4]:
# Day 6 - Step 2: Recreate Train/Test Preprocessing

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Separate features and target
X = base_df.drop(columns=[TARGET])
y = base_df[TARGET]

# Remove the constant feature identified during Day 1-3
constant_features = [
    col for col in X.columns
    if X[col].nunique() <= 1
]

X = X.drop(columns=constant_features)

# Stratified train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Feature types
categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_features = X_train.select_dtypes(
    exclude=["object"]
).columns.tolist()

# One-hot encode categorical features
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

encoder.fit(X_train[categorical_features])

X_train_cat = encoder.transform(
    X_train[categorical_features]
)

X_test_cat = encoder.transform(
    X_test[categorical_features]
)

# Numerical features
X_train_num = X_train[numerical_features].to_numpy()
X_test_num = X_test[numerical_features].to_numpy()

# Final input for tree-based models
X_train_processed = np.hstack([
    X_train_num,
    X_train_cat
])

X_test_processed = np.hstack([
    X_test_num,
    X_test_cat
])

# Scaled version for Logistic Regression consistency
scaler = StandardScaler()

X_train_num_scaled = scaler.fit_transform(
    X_train[numerical_features]
)

X_test_num_scaled = scaler.transform(
    X_test[numerical_features]
)

X_train_final = np.hstack([
    X_train_num_scaled,
    X_train_cat
])

X_test_final = np.hstack([
    X_test_num_scaled,
    X_test_cat
])

print("PREPROCESSING READY")
print("=" * 50)
print("Training data      :", X_train.shape)
print("Testing data       :", X_test.shape)
print("Numerical features :", len(numerical_features))
print("Categorical features:", len(categorical_features))
print("Encoded categories :", X_train_cat.shape[1])
print("Final HGB features :", X_train_processed.shape[1])
print("Train fraud cases  :", y_train.sum())
print("Test fraud cases   :", y_test.sum())

C:\Users\prala\AppData\Local\Temp\ipykernel_1928\2509253475.py:28: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


PREPROCESSING READY
Training data      : (800000, 30)
Testing data       : (200000, 30)
Numerical features : 25
Categorical features: 5
Encoded categories : 26
Final HGB features : 51
Train fraud cases  : 8823
Test fraud cases   : 2206


In [5]:
# Day 6 - Step 3: Rebuild Final Tuned Model

from sklearn.ensemble import HistGradientBoostingClassifier

tuned_hgb_model = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    random_state=42
)

print("Training final model...")
tuned_hgb_model.fit(
    X_train_processed,
    y_train
)

print("\nFinal model trained successfully.")
print("Model:", type(tuned_hgb_model).__name__)
print("Iterations:", tuned_hgb_model.n_iter_)

Training final model...

Final model trained successfully.
Model: HistGradientBoostingClassifier
Iterations: 300


In [6]:
# Day 6 - Step 4: Verify Final Model Performance

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# Generate fraud probabilities
tuned_hgb_y_pred_proba = tuned_hgb_model.predict_proba(
    X_test_processed
)[:, 1]

# Apply the selected threshold
final_threshold = 0.10

final_y_pred = (
    tuned_hgb_y_pred_proba >= final_threshold
).astype(int)

# Calculate metrics
final_accuracy = accuracy_score(y_test, final_y_pred)
final_precision = precision_score(y_test, final_y_pred)
final_recall = recall_score(y_test, final_y_pred)
final_f1 = f1_score(y_test, final_y_pred)
final_roc_auc = roc_auc_score(y_test, tuned_hgb_y_pred_proba)
final_pr_auc = average_precision_score(y_test, tuned_hgb_y_pred_proba)
final_cm = confusion_matrix(y_test, final_y_pred)

print("FINAL MODEL VERIFICATION")
print("=" * 60)
print("Model     :", type(tuned_hgb_model).__name__)
print("Threshold :", final_threshold)
print()
print("Accuracy  :", round(final_accuracy, 4))
print("Precision :", round(final_precision, 4))
print("Recall    :", round(final_recall, 4))
print("F1 Score  :", round(final_f1, 4))
print("ROC-AUC   :", round(final_roc_auc, 4))
print("PR-AUC    :", round(final_pr_auc, 4))
print()
print("Confusion Matrix:")
print(final_cm)

FINAL MODEL VERIFICATION
Model     : HistGradientBoostingClassifier
Threshold : 0.1

Accuracy  : 0.9778
Precision : 0.1904
Recall    : 0.3128
F1 Score  : 0.2367
ROC-AUC   : 0.8961
PR-AUC    : 0.1698

Confusion Matrix:
[[194861   2933]
 [  1516    690]]


In [7]:
# Day 6 - Step 5: Save Final Model Bundle

import joblib

model_bundle = {
    "model": tuned_hgb_model,
    "encoder": encoder,
    "scaler": scaler,
    "categorical_features": categorical_features,
    "numerical_features": numerical_features,
    "constant_features": constant_features,
    "target": TARGET,
    "threshold": final_threshold,
    "model_name": "Tuned HistGradientBoosting"
}

MODEL_FILE = "../fraud_detection_model_bundle.joblib"

joblib.dump(
    model_bundle,
    MODEL_FILE
)

print("Final model bundle saved successfully.")
print("File:", MODEL_FILE)

Final model bundle saved successfully.
File: ../fraud_detection_model_bundle.joblib


In [8]:
# Day 6 - Step 6: Verify Saved Model Bundle

import os
import joblib

MODEL_FILE = "../fraud_detection_model_bundle.joblib"

print("File exists:", os.path.exists(MODEL_FILE))

# Load the saved bundle
saved_bundle = joblib.load(MODEL_FILE)

print("\nSaved model bundle loaded successfully.")
print("Model       :", type(saved_bundle["model"]).__name__)
print("Threshold   :", saved_bundle["threshold"])
print("Target      :", saved_bundle["target"])
print("Numerical features :", len(saved_bundle["numerical_features"]))
print("Categorical features:", len(saved_bundle["categorical_features"]))

File exists: True

Saved model bundle loaded successfully.
Model       : HistGradientBoostingClassifier
Threshold   : 0.1
Target      : fraud_bool
Numerical features : 25
Categorical features: 5


In [9]:
# Day 6 - Step 7: Fraud Risk Score & Risk Levels

# Model fraud probability
fraud_probability = tuned_hgb_y_pred_proba

# Convert probability to a 0-100 risk score
risk_score = fraud_probability * 100

# Define risk levels
def assign_risk_level(score):
    if score < 10:
        return "Low"
    elif score < 30:
        return "Medium"
    elif score < 60:
        return "High"
    else:
        return "Critical"

risk_level = np.array([
    assign_risk_level(score)
    for score in risk_score
])

# Create risk summary
risk_summary = pd.Series(
    risk_level
).value_counts().reindex(
    ["Low", "Medium", "High", "Critical"],
    fill_value=0
)

print("FRAUD RISK SCORING")
print("=" * 50)
print("Risk score range :", round(risk_score.min(), 2), "to", round(risk_score.max(), 2))
print("Decision threshold:", final_threshold * 100, "risk score")
print()
print("Risk Level Distribution")
print("-" * 50)
print(risk_summary)

FRAUD RISK SCORING
Risk score range : 0.01 to 94.11
Decision threshold: 10.0 risk score

Risk Level Distribution
--------------------------------------------------
Low         196377
Medium        3025
High           514
Critical        84
Name: count, dtype: int64


In [10]:
# Day 6 - Step 8: Create and Save Risk Results

risk_results = X_test.copy()

risk_results["Actual_Fraud"] = y_test.to_numpy()
risk_results["Fraud_Probability"] = fraud_probability
risk_results["Risk_Score"] = risk_score
risk_results["Predicted_Fraud"] = final_y_pred
risk_results["Risk_Level"] = risk_level

RISK_FILE = "../fraud_risk_results.csv"

risk_results.to_csv(
    RISK_FILE,
    index=False
)

print("Risk results saved successfully.")
print("File:", RISK_FILE)
print("Rows:", len(risk_results))
print("Columns:", len(risk_results.columns))

Risk results saved successfully.
File: ../fraud_risk_results.csv
Rows: 200000
Columns: 35


In [11]:
# Day 6 - Step 9: Verify Risk Results

print("RISK RESULTS VERIFICATION")
print("=" * 60)

print("Shape:", risk_results.shape)
print()

print("Required columns:")
required_columns = [
    "Actual_Fraud",
    "Fraud_Probability",
    "Risk_Score",
    "Predicted_Fraud",
    "Risk_Level"
]

for col in required_columns:
    print(f"{col:20}:", col in risk_results.columns)

print()
print("Actual fraud cases    :", risk_results["Actual_Fraud"].sum())
print("Predicted fraud cases :", risk_results["Predicted_Fraud"].sum())
print()

print("Risk level counts:")
print(risk_results["Risk_Level"].value_counts().reindex(
    ["Low", "Medium", "High", "Critical"],
    fill_value=0
))

print()
print("Risk score range      :",
      round(risk_results["Risk_Score"].min(), 2),
      "to",
      round(risk_results["Risk_Score"].max(), 2))

print()
print("Verification complete.")

RISK RESULTS VERIFICATION
Shape: (200000, 35)

Required columns:
Actual_Fraud        : True
Fraud_Probability   : True
Risk_Score          : True
Predicted_Fraud     : True
Risk_Level          : True

Actual fraud cases    : 2206
Predicted fraud cases : 3623

Risk level counts:
Risk_Level
Low         196377
Medium        3025
High           514
Critical        84
Name: count, dtype: int64

Risk score range      : 0.01 to 94.11

Verification complete.
